### Master Calendar Generator
This notebook allows you to upload an Excel file, automatically detects all date columns, and generates a master calendar ranging from the earliest to the latest date found in your data.

In [ ]:
import pandas as pd
import numpy as np

import os
from google.colab import files

# 1. Setup Folders
for folder in ['DataBase', 'Output']:
    if not os.path.exists(folder):
        os.makedirs(folder)
        print(f"Created folder: {folder}")

# 2. Upload Excel File
print("Please upload your Excel file:")
uploaded = files.upload()
if not uploaded:
    print("No file uploaded.")
else:
    file_name = list(uploaded.keys())[0]
    db_path = os.path.join('DataBase', file_name)
    with open(db_path, 'wb') as f:
        f.write(uploaded[file_name])

    # 3. Read Data and Identify Strictly Datetime Columns
    df = pd.read_excel(db_path)
    date_columns = []

    print("\nAnalyzing columns for strict datetime formats...")
    for col in df.columns:
        # Skip purely numeric columns (common cause of false positives)
        if pd.api.types.is_numeric_dtype(df[col]):
            continue
            
        # If already datetime, add it
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            date_columns.append(col)
        else:
            try:
                # Attempt conversion on string/object columns
                temp_dates = pd.to_datetime(df[col], errors='coerce')
                # Valid if more than 50% are dates and they aren't all just the Unix epoch (1970)
                if temp_dates.notna().sum() / len(df) > 0.5:
                    if temp_dates.min() > pd.Timestamp('1980-01-01'):
                        df[col] = temp_dates
                        date_columns.append(col)
            except:
                continue

    print(f"Strictly Identified Date Columns: {date_columns}")

    # 4. Determine Global Date Range
    if not date_columns:
        print("No valid date columns found. Creating a default calendar for the current year.")
        start_date = pd.Timestamp.now().replace(month=1, day=1)
        end_date = pd.Timestamp.now().replace(month=12, day=31)
    else:
        all_dates = pd.concat([df[col] for col in date_columns])
        start_date = all_dates.min()
        end_date = all_dates.max()

    print(f"Calendar Range: {start_date.date()} to {end_date.date()}")

    # 5. Create Master Calendar
    calendar_df = pd.DataFrame({'Date': pd.date_range(start=start_date, end=end_date, freq='D')})

    calendar_df['Year'] = calendar_df['Date'].dt.year
    calendar_df['Month'] = calendar_df['Date'].dt.month
    calendar_df['Month_Name'] = calendar_df['Date'].dt.month_name()
    calendar_df['Day'] = calendar_df['Date'].dt.day
    calendar_df['Day_Name'] = calendar_df['Date'].dt.day_name()
    calendar_df['Day_of_Week'] = calendar_df['Date'].dt.dayofweek
    calendar_df['Week_of_Year'] = calendar_df['Date'].dt.isocalendar().week
    calendar_df['Quarter'] = calendar_df['Date'].dt.quarter
    calendar_df['Is_Weekend'] = calendar_df['Date'].dt.dayofweek >= 5

    # 6. Save Output
    output_path = os.path.join('Output', 'MasterCalendar.xlsx')
    calendar_df.to_excel(output_path, index=False)

    print(f"\nSuccess! Master Calendar saved to: {output_path}")
    display(calendar_df.head())